# End-to-End Cluster Execution Flow: From Code to Tasks

**Module**: 01_spark_fundamentals  
**Notebook**: 04_spark_cluster_execution.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: The step-by-step lifecycle of an Apache Spark job: from code declaration to DAG construction, stage splitting, task scheduling, and result return.
- **Why does this topic exist?**: To diagnose bottlenecks, you must be able to trace exactly where a query is spending time: DAGScheduler, TaskScheduler, or network shuffle.
- **What problem does it solve?**: Without understanding execution hierarchy (Action -> Job -> Stage -> Task), developers cannot decipher the Spark UI.
- **Where is it used in real-world Data Engineering?**: Explaining execution flow and debugging stuck stages are top-tier Senior Data Engineering interview topics.

---

## 2. Complete Execution Hierarchy

```
[User Action: df.count()]
        |
        v
+-----------------------------------------------------------+
| JOB (1 per Action)                                        |
|  Identified by Job ID (e.g. Job 0, Job 1)                 |
+-----------------------------+-----------------------------+
                              |
                              v DAGScheduler splits at Shuffles
+-----------------------------------------------------------+
| STAGES (Bounded by Wide Dependencies)                     |
|  - Stage 0: ShuffleMapStage (reads, filters, map output)  |
|  - Stage 1: ResultStage (fetches shuffle, aggregates)     |
+-----------------------------+-----------------------------+
                              |
                              v TaskScheduler assigns to Cores
+-----------------------------------------------------------+
| TASKS (1 per Partition in Stage)                          |
|  - Task 0 -> Core 1 on Worker 1                           |
|  - Task 1 -> Core 2 on Worker 1                           |
|  - Task 2 -> Core 1 on Worker 2                           |
|  - Task 3 -> Core 2 on Worker 2                           |
+-----------------------------------------------------------+
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Execution_Flow_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Practical: Generating a Multi-Stage Job

Let us run an operation with an explicit wide dependency (shuffle) and trace its job and stage breakdown.

In [ ]:
orders_df = spark.read.option("header", True).option("inferSchema", True).csv("/opt/spark-data/retail/orders.csv")

# A 2-stage execution: Filter (narrow) + GroupBy (wide)
agg_df = (
    orders_df
    .filter("status = 'Completed'")
    .groupBy("channel")
    .agg(
        F.count("order_id").alias("order_count"),
        F.round(F.sum(F.col("quantity") * F.col("unit_price")), 2).alias("total_revenue")
    )
)

print("=== Execution Plan ===")
agg_df.explain("formatted")

## 4. Triggering the Action and Checking the Spark UI

Now we trigger the Action (`show()`) and inspect the generated Job in the Spark UI (`http://localhost:4040`).

In [ ]:
start_time = time.time()
agg_df.show()
elapsed = time.time() - start_time
print(f"Action executed in: {elapsed:.3f} seconds")
print("Check http://localhost:4040/jobs/ to inspect Job, Stages, and Tasks metrics!")

---

## Experiment: Predicting and Verifying Stage Count

- **1. What are we testing?**: Verify that a query with 2 wide transformations (groupBy + repartition) produces exactly 3 stages.
- **2. What do we expect?**: The DAG will have 3 stages separated by 2 shuffle exchange boundaries.

In [ ]:
# Query with 2 shuffles: groupBy followed by repartition
multi_shuffle = (
    orders_df
    .groupBy("customer_id")
    .count()
    .repartition(4)
)

# Observe Exchange nodes in plan
multi_shuffle.explain()
print(f"Result count: {multi_shuffle.count()}")

**3. What happened?**: Spark UI shows 3 stages for this job (Scan & partial group, shuffle merge & recount, repartition exchange).  
**4. Why did it happen?**: Each wide dependency (Exchange node) creates a new Stage boundary.  
**5. Practical Engineering Lesson**: Counting Exchange nodes in explain() allows you to predict the exact number of stages in the Spark UI.

---

## Practical Exercises

### Exercise 1 (Beginner): Identify Narrow Transformation Chain
**Problem Statement**: Write a pipeline chaining filter, withColumn, and select. How many stages will count() produce?  
**Input / Context**: orders_df

In [ ]:
# Exercise 1: Count stages for narrow pipeline

### Exercise 2 (Intermediate): Trace Task Count from Partition Count
**Problem Statement**: Set shuffle partitions to 5, execute a groupBy count, and check the task count in the final stage.  
**Input / Context**: spark.conf.set('spark.sql.shuffle.partitions','5')

In [ ]:
# Exercise 2: Tune and verify task count

### Exercise 3 (Advanced): End-to-End Latency Breakdown
**Problem Statement**: Measure wall-clock latency of reading 60k orders, performing aggregation, and writing to Parquet.  
**Input / Context**: retail/orders.csv

In [ ]:
# Exercise 3: Full pipeline timing

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Identify Narrow Transformation Chain):
```python
narrow_df = orders_df.filter("quantity > 1").withColumn("tax", F.col("unit_price")*0.1).select("order_id","tax")
# Has ZERO Exchange nodes -> exactly 1 Stage!
narrow_df.explain()
print("Count:", narrow_df.count())
```

#### Solution 2 (Trace Task Count from Partition Count):
```python
spark.conf.set("spark.sql.shuffle.partitions","5")
res = orders_df.groupBy("status").count()
print("Number of partitions in aggregated DF:", res.rdd.getNumPartitions())
res.show()
spark.conf.set("spark.sql.shuffle.partitions","8")
```

#### Solution 3 (End-to-End Latency Breakdown):
```python
out_path ="/opt/spark-data/warehouse/test_perf_out"
t0 = time.time()
(
    orders_df
    .filter("status ='Completed'")
    .groupBy("product_id")
    .agg(F.sum("quantity").alias("total_qty"))
    .coalesce(1)
    .write.mode("overwrite").parquet(out_path)
)
print(f"Full pipeline executed and written to Parquet in {time.time()-t0:.2f}s")
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Hierarchy**: Action triggers Job -> Job splits into Stages at shuffles -> Stages split into Tasks per partition.
- **Pipelining**: Narrow operations execute together in a single stage without intermediate disk or network hops.
- **Task Execution**: Each task runs on exactly one partition on an executor thread.

### Essential Technical Terms
- `Job`: Execution triggered by one Action.
- `Stage`: Pipelined set of tasks separated by shuffle dependencies.
- `Task`: Individual thread unit processing a single partition.

### Common Engineering Mistakes
- **Expecting Stages for Every Operation**: Only wide transformations (shuffles) create stage boundaries; narrow operations are fused.
- **Ignoring Task Count**: Too few tasks leaves cores idle; too many tasks floods scheduler.

### Interview Questions on This Topic
- **Q: What determines the number of tasks in a stage?**
  - *A*: The number of partitions in the RDD/DataFrame entering that stage.
- **Q: What causes a new stage to be created?**
  - *A*: A shuffle dependency (wide transformation) such as groupBy, join, or repartition.

### What to Study Next
Module 01 complete! Proceed to Module 02: Resilient Distributed Datasets (RDDs).

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')